# QuantJourney SDK - SEC Filing Fundamental Price Mosaic

This notebook demonstrates a QuantJourney SDK workflow that joins SEC filings, company facts, FMP statements, ratios, analyst estimates, identity and price-volume reaction for one issuer.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbol = 'AAPL'
filings_raw = qj.sec.get_company_filings(symbol=symbol, limit=40)
facts_raw = qj.sec.get_company_facts(symbol=symbol)
submissions_raw = qj.sec.get_company_submissions(symbol=symbol)
income_raw = qj.fmp.get_income_statement(symbol=symbol, period='annual', limit=5)
balance_raw = qj.fmp.get_balance_sheet_statement(symbol=symbol, period='annual', limit=5)
ratios_raw = qj.fmp.get_financial_ratios_ttm(symbol=symbol)
estimates_raw = qj.fmp.get_analyst_estimates(symbol=symbol, period='annual', limit=8)
identity_raw = qj.openfigi.get_figi_data(symbol=symbol, exchange='US')
prices = price_frame(symbol, start='2021-01-01', end=END)


In [ ]:
filings = pd.DataFrame(as_rows(filings_raw))
income = pd.DataFrame(as_rows(income_raw))
balance = pd.DataFrame(as_rows(balance_raw))
estimates = pd.DataFrame(as_rows(estimates_raw))
ratios = pd.DataFrame(as_rows(ratios_raw))
identity = pd.DataFrame(as_rows(identity_raw))
if filings.empty:
    raise RuntimeError('No SEC filing rows returned')


In [ ]:
event_dates = pd.to_datetime(filings.get('filingDate', filings.get('filedAt', filings.get('date'))), errors='coerce').dropna()
reactions = []
for event_date in event_dates.head(20):
    pos = prices.index.searchsorted(event_date)
    if pos > 5 and pos + 5 < len(prices):
        pre = prices['price'].iloc[pos - 5]
        post = prices['price'].iloc[pos + 5]
        reactions.append({'filing_date': event_date, 'five_day_reaction': post / pre - 1})
reaction = pd.DataFrame(reactions)
summary = pd.Series({'sec_filings': len(filings), 'company_fact_rows': len(as_rows(facts_raw)), 'income_statement_rows': len(income), 'balance_sheet_rows': len(balance), 'estimate_rows': len(estimates), 'identity_rows': len(identity), 'median_5d_filing_reaction': reaction['five_day_reaction'].median() if not reaction.empty else np.nan})
display(summary)
display(filings.head())
prices['price'].tail(756).plot(title=f'{symbol} price with SEC filing events')
for event_date in event_dates.head(12):
    plt.axvline(event_date, color='tab:pink', alpha=0.35)
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.